In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#URL = 'https://github.com/aiedu-courses/stepik_eda_and_dev_tools/blob/main/datasets/abalone.csv'
URL = 'abalone.csv'
df = pd.read_csv(URL)
df.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,NaN,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


**Очистка, данных**

In [4]:
# Опечатки можно заменить сразу на исходном датасете
df['Sex'] = df['Sex'].replace('f', 'F')
df.Sex.value_counts()

Sex
F    1454
M    1447
I    1276
Name: count, dtype: int64

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# класс для заполнения пропусков
# для избежания утечки данных будет выполнен после разбиения на train-test
missing_filler = ColumnTransformer(
    [('imputer', SimpleImputer(strategy='median'), ['Whole weight', 'Diameter', 'Shell weight'])],
    remainder='passthrough',
    verbose_feature_names_out=False
)
missing_filler.set_output(transform="pandas")

missing_filler.fit_transform(df).isna().sum()

Whole weight      0
Diameter          0
Shell weight      0
Sex               0
Length            0
Height            0
Shucked weight    0
Viscera weight    0
Rings             0
dtype: int64

**Модель GaussianNB**

In [7]:
y = df['Rings']
X = df.drop(['Rings', 'Sex'], axis=1)

#y.value_counts().sort_values()

In [8]:
#выделим 2 класса Rings < 10 и Rings >= 10
y_class = (y >= 10).astype(int)

In [9]:
y_class.value_counts()

Rings
0    2096
1    2081
Name: count, dtype: int64

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_class, test_size=0.25, random_state=42)
X_train = missing_filler.fit_transform(X_train)
X_test = missing_filler.transform(X_test)

In [11]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

In [12]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.7301435406698564

In [13]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

confusion_matrix(y_test, y_pred)

array([[387, 147],
       [135, 376]])

Бинарная классификация по возрасту с помощью GaussianNB показала точность 73% при хорошем балансе классов в исходных данных. Модель демонстрирует стабильные результаты, допуская примерно равное количество ошибок для обеих возрастных групп (молодых и зрелых особей). Полученный показатель позволяет считать данный подход адекватным для базового решения задачи разделения моллюсков по порогу в 10 колец.

**KNN**

In [16]:
from sklearn.neighbors import KNeighborsClassifier

knn_cl = KNeighborsClassifier()
knn_cl.fit(X_train, y_train)
pred_knn = knn_cl.predict(X_test)

In [17]:
accuracy_score(y_test, pred_knn)

0.7732057416267942

In [18]:
confusion_matrix(y_test, pred_knn)

array([[417, 117],
       [120, 391]])

Использование алгоритма KNN позволило достичь точности 77%, что на 4% выше результата модели GaussianNB. Метод ближайших соседей оказался эффективнее, так как он лучше учитывает сложные нелинейные связи между физическими параметрами моллюсков. Анализ матрицы ошибок подтверждает, что модель стала реже ошибаться в обеих группах, сохраняя при этом общую стабильность предсказаний.

**Подбор гиперпараметров GridSearchCV**

In [21]:
from sklearn.model_selection import GridSearchCV

model = KNeighborsClassifier()
params = {'n_neighbors' : np.arange(2, 20, 2),
          'weights' : ['uniform', 'distance'],
          'p' : [1, 2]}

gs = GridSearchCV(model, params, scoring='accuracy', cv=3, n_jobs=-1, verbose=2)
gs.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'n_neighbors': array([ 2, 4..., 14, 16, 18]), 'p': [1, 2], 'weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 :

In [22]:
gs.best_score_, gs.best_params_

(np.float64(0.7857598978288634),
 {'n_neighbors': np.int64(12), 'p': 2, 'weights': 'distance'})

In [23]:
pred = gs.best_estimator_.predict(X_test)

accuracy_score(y_test, pred)

0.784688995215311

In [24]:
confusion_matrix(y_test, pred)

array([[417, 117],
       [108, 403]])

Подбор гиперпараметров через GridSearchCV позволил найти оптимальную конфигурацию (12 соседей, веса 'distance'), что повысило точность модели до 78.5%. Минимальный разрыв между результатами кросс-валидации и теста подтверждает высокую стабильность модели и отсутствие переобучения. На данный момент это лучший результат, обеспечивающий наиболее качественное разделение моллюсков по возрастному признаку.

**Добавление категориальных признаков**

In [27]:
X_full = df.drop('Rings', axis=1)
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_full, y_class, test_size=0.25, random_state=42)
X_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   str    
 1   Length          4177 non-null   float64
 2   Diameter        4078 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole weight    4078 non-null   float64
 5   Shucked weight  4177 non-null   float64
 6   Viscera weight  4177 non-null   float64
 7   Shell weight    4127 non-null   float64
dtypes: float64(7), str(1)
memory usage: 261.2 KB


In [28]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline

numerical_features = ['Length','Diameter','Height','Whole weight','Shucked weight','Viscera weight','Shell weight']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaling', MinMaxScaler())
])

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first', handle_unknown="ignore"), ['Sex']),
    ('num', num_pipe, numerical_features)
])

ct

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [29]:
X_train_transformed = ct.fit_transform(X_train_full)
X_test_transformed = ct.transform(X_test_full)

X_train_transformed.shape, X_test_transformed.shape

((3132, 9), (1045, 9))

In [30]:
new_features = list(ct.named_transformers_['ohe'].get_feature_names_out())
new_features.extend(numerical_features)
new_features, len(new_features)

(['Sex_I',
  'Sex_M',
  'Length',
  'Diameter',
  'Height',
  'Whole weight',
  'Shucked weight',
  'Viscera weight',
  'Shell weight'],
 9)

In [31]:
X_train_transformed = pd.DataFrame(X_train_transformed, columns=new_features)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=new_features)

X_test_transformed.head()

,Sex_I,Sex_M,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight
0,0.0,1.0,0.716216,0.672269,0.141593,0.390119,0.282448,0.396313,0.322372
1,0.0,1.0,0.695946,0.647059,0.132743,0.308305,0.259583,0.282423,0.242651
2,0.0,0.0,0.655405,0.655462,0.172566,0.346733,0.204438,0.294931,0.332337
3,0.0,0.0,0.756757,0.731092,0.150442,0.446078,0.361466,0.350230,0.377180
4,0.0,1.0,0.540541,0.554622,0.128319,0.217992,0.157364,0.141540,0.212755


In [32]:
model = KNeighborsClassifier()

params = {'n_neighbors' : np.arange(2, 20, 2),
          'weights' : ['uniform', 'distance']}

In [33]:
gs = GridSearchCV(model, params, scoring='accuracy', cv=3, n_jobs=-1, verbose=5)
gs.fit(X_train_transformed, y_train_full)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'n_neighbors': array([ 2, 4..., 14, 16, 18]), 'weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is

In [34]:
gs.best_score_, gs.best_params_

(np.float64(0.780332056194125),
 {'n_neighbors': np.int64(18), 'weights': 'distance'})

In [35]:
pred = gs.best_estimator_.predict(X_test_transformed)

accuracy_score(y_test_full, pred)

0.7760765550239235

In [36]:
confusion_matrix(y_test, pred)

array([[409, 125],
       [109, 402]])

Добавление категориального признака и масштабирование не улучшили качество модели — точность даже немного снизилась с 78.47% до 77.61% (разница около −0.9 п.п.).
Анализ матрицы ошибок показывает небольшие изменения во внутреннем поведении модели: она стала чуть хуже распознавать оба класса. Для первого класса количество верных предсказаний снизилось с 417 до 409, а число ошибок выросло. Для второго класса также наблюдается незначительное ухудшение: число верных предсказаний уменьшилось с 403 до 402, при этом ошибки немного увеличились.
В целом, усложнение пайплайна с помощью добавления категориального признака и масштабирования не дало положительного эффекта для алгоритма KNN.

**Построение Explainer Dashboard**

In [39]:
from explainerdashboard import ClassifierExplainer, ExplainerDashboard
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [40]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

explainer = ClassifierExplainer(gs.best_estimator_, X_test_transformed, y_test_full)
db = ExplainerDashboard(explainer)

  0%|          | 0/1045 [00:00<?, ?it/s]

In [41]:
db.run(host='127.0.0.1', port=8050)

1. Наиболее важные факторы в среднем для получения прогноза
Shell weight (Вес раковины) — самый значимый признак, оказывающий наибольшее влияние на предсказание.
Shucked weight (Вес мяса) — второй по важности фактор.
Whole weight (Общий вес).
Sex_I (Пол: инфант/неопределенный).
Viscera weight (Вес внутренностей).
Такие признаки, как Height (Высота), Sex_M (Пол: мужской) и Length (Длина), оказывают наименьшее влияние на модель в среднем.

2. Значения метрик и их интерпретация
Модель показала следующие метрики производительности:
Accuracy (Доля правильных ответов) = 0.776: В 77.6% случаев модель верно предсказывает класс.
Precision (Точность) = 0.763: Из всех объектов, которые модель отнесла к положительному классу, 76.3% действительно являются таковыми.
Recall (Полнота) = 0.787: Модель успешно находит и классифицирует 78.7% от всех реально существующих положительных объектов.
F1-score = 0.775: Гармоническое среднее между точностью и полнотой. Значение близко к Precision и Recall, что говорит о хорошем балансе модели и отсутствии сильного перекоса.
ROC AUC = 0.871: Площадь под ROC-кривой. Значение 0.871 говорит о высокой разделяющей способности модели — она с вероятностью 87.1% поставит случайно выбранный положительный объект выше, чем случайно выбранный отрицательный.
PR AUC = 0.866: Площадь под кривой Precision-Recall. Высокое значение подтверждает, что модель хорошо справляется с задачей даже с учетом возможного дисбаланса классов.
Log Loss (Логистическая функция потерь) = 0.502: Штрафует модель за неуверенные и неверные вероятностные предсказания. Значение 0.502 свидетельствует об умеренной уверенности предсказаний.

3. Анализ индивидуальных прогнозов
Графики SHAP (SHapley Additive exPlanations)

Пример 1: (Индекс 129)
Shell weight: 0.466
Shucked weight: 0.332
Суммарный SHAP: 0.296

Комментарий: У данного экземпляра значения веса раковины и веса мяса значительно выше среднего. Это дает сильный положительный вклад. Модель уверенно сдвигает прогноз в сторону положительного класса из-за больших физических габаритов.

Пример 2: (Индекс 134)
Shell weight: 0.029
Shucked weight: 0.024
Суммарный SHAP: -0.143

Комментарий: Здесь ситуация обратная — значения веса крайне малы. Эти факторы тянут прогноз вниз, уменьшая вероятность принадлежности к положительному классу. Модель классифицирует этот экземпляр как отрицательный именно на основе этих низких показателей веса.

Пример 3: (Индекс 124)
Shell weight: 0.158
Shucked weight: 0.124
Суммарный SHAP: -0.093

Комментарий: Значения веса у этого экземпляра ниже среднего, но больше, чем в Примере 2. Они по-прежнему дают отрицательный вклад в прогноз, но с меньшей силой отталкивают вероятность от положительного класса.